In [3]:
import os
import sys

# Point to the exact python executable inside your active conda environment
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

PROJECT_ROOT = r"D:\DataBricks\trading_scanner"

if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)


In [13]:
import yfinance as yf
import pandas as pd
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()


In [22]:
df_5m = yf.download(tickers="RELIANCE.NS", period="1d", interval="5m")

[*********************100%***********************]  1 of 1 completed


In [23]:
import pyspark.sql.functions as F

# 1. Move the pandas index into a regular column named 'Datetime'
df_5m_reset = df_5m.reset_index()

# 2. Convert the updated pandas DataFrame to Spark
df = spark.createDataFrame(df_5m_reset)

# 3. Clean up the Spark column names (replaces tuples/multi-index names with clean strings)
df = df.toDF("Datetime", "Close", "High", "Low", "Open", "Volume")

# 4. Convert the Datetime column to 'Asia/Kolkata' timezone
df_with_ist = df.withColumn("Local_Time", F.from_utc_timestamp(F.col("Datetime"), "Asia/Kolkata"))    #.withColumn("BARS", F.when(F.col("Open") < F.col("Close"),"GREEN").otherwise("RED"))

In [24]:
from pyspark.sql.functions.builtin import greatest
df = df_with_ist["Local_Time", "Close", "High", "Low", "Open", "Volume"]
df_feature_engineering = df.withColumn("Body",F.abs(F.col("Open")-F.col("Close")))\
.withColumn("Upper_wick",F.col("High")-F.greatest(F.col("Open"),F.col("Close")))\
.withColumn("Lower_wick",F.least(F.col("Open"),F.col("Close"))-F.col("Low"))\
.withColumn("Total_Range",F.abs(F.col("High")-F.col("Low")))

In [25]:
df_feature_engineering.show(10)

+-------------------+------------------+------------------+------------------+------------------+------+---------------+---------------+---------------+---------------+
|         Local_Time|             Close|              High|               Low|              Open|Volume|           Body|     Upper_wick|     Lower_wick|    Total_Range|
+-------------------+------------------+------------------+------------------+------------------+------+---------------+---------------+---------------+---------------+
|2026-07-01 14:45:00| 1298.800048828125|1299.9000244140625|1296.9000244140625|            1297.5|143052| 1.300048828125|1.0999755859375|0.5999755859375|            3.0|
|2026-07-01 14:50:00|1299.5999755859375| 1299.699951171875|1297.9000244140625|            1299.0|105173|0.5999755859375|0.0999755859375|1.0999755859375|1.7999267578125|
|2026-07-01 14:55:00| 1301.300048828125| 1301.300048828125| 1298.800048828125|1299.5999755859375| 91957|1.7000732421875|            0.0|0.7999267578125|   

In [24]:
from patterns.hammer import detect_hammer

df = detect_hammer(df_feature_engineering)
df.filter(F.col("is_hammer")).show(10)

+-------------------+-----------------+------------------+------------------+------------------+------+---------------+----------+---------------+---------------+---------+
|         Local_Time|            Close|              High|               Low|              Open|Volume|           Body|Upper_wick|     Lower_wick|    Total_Range|is_hammer|
+-------------------+-----------------+------------------+------------------+------------------+------+---------------+----------+---------------+---------------+---------+
|2026-06-24 16:05:00|1311.800048828125|1312.0999755859375|            1311.0|1312.0999755859375| 47703|0.2999267578125|       0.0| 0.800048828125|1.0999755859375|     true|
|2026-06-25 20:30:00|1319.199951171875|1319.9000244140625|1317.4000244140625|1319.9000244140625|861810|0.7000732421875|       0.0|1.7999267578125|            2.5|     true|
+-------------------+-----------------+------------------+------------------+------------------+------+---------------+----------+-----

In [26]:
%load_ext autoreload
%autoreload 2

from config import *
import importlib
from data.loader import load_market_data
from data.normalizer import normalize_market_data
from utils.datetime_utils import convert_utc_to_ist

#importlib.reload(data.normalizer)

df = load_market_data(SYMBOLS,PERIOD,INTERVAL)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


[*********************100%***********************]  30 of 30 completed


In [27]:
normalized_df = normalize_market_data(df)
normalized_df = convert_utc_to_ist(normalized_df)

D:\DataBricks\trading_scanner\data\normalizer.py:3: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  df = df.stack()


In [29]:
normalized_df

,Symbol,Datetime,Open,High,Low,Close,Volume,Trade_Date
0,ADANIENT,2026-07-01 09:15:00+05:30,3026.100098,3027.500000,3016.000000,3020.000000,24378,2026-07-01
1,ADANIPORTS,2026-07-01 09:15:00+05:30,1825.000000,1827.500000,1815.800049,1817.800049,73683,2026-07-01
2,ASIANPAINT,2026-07-01 09:15:00+05:30,2647.600098,2647.600098,2624.000000,2624.199951,22013,2026-07-01
3,AXISBANK,2026-07-01 09:15:00+05:30,1354.699951,1354.699951,1346.400024,1352.000000,41766,2026-07-01
4,BAJAJ-AUTO,2026-07-01 09:15:00+05:30,9780.000000,9812.000000,9780.000000,9801.500000,6926,2026-07-01
...,...,...,...,...,...,...,...,...
115,SBIN,2026-07-01 09:30:00+05:30,1026.800049,1027.500000,1026.599976,1026.599976,29228,2026-07-01
116,SUNPHARMA,2026-07-01 09:30:00+05:30,1873.800049,1876.800049,1873.800049,1874.699951,27494,2026-07-01
117,TCS,2026-07-01 09:30:00+05:30,2048.300049,2049.399902,2045.199951,2045.199951,13984,2026-07-01
118,TITAN,2026-07-01 09:30:00+05:30,4485.799805,4490.000000,4479.000000,4481.399902,11208,2026-07-01


In [8]:
import datetime as dt
import pandas as pd
from storage.market_store import build_storage_path
symbol_list = normalized_df["Symbol"].unique().tolist()
print(symbol_list[0])
current_date = normalized_df["Trade_Date"].unique()[0]
store_path = build_storage_path(symbol_list[0],INTERVAL,current_date)

print(store_path)

ADANIENT
D:\DataBricks\trading_scanner\market_data\ADANIENT\5m\2026\06\2026_06_30.parquet


In [21]:
import datetime

current_date = datetime.date
print(current_date)

<class 'datetime.date'>
